In [10]:
## Import Libraries
import io
import pandas as pd
import yfinance as yf
import requests

In [11]:
def fetch_tickers(url="", table=0, index_col="Symbol", company_col="Security",  cik_col="CIK"):
    # Fetch with proper User-Agent header to avoid 403 error
    headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
    response = requests.get(url, headers=headers)
    tables = pd.read_html(io.StringIO(response.text))
    index_table = tables[table]  # Use the specified table index
    tickers = index_table[index_col].tolist()
    companies = index_table[company_col].tolist() if company_col in index_table.columns else None
    # cik = index_table[cik_col].tolist() if cik_col in index_table.columns else Nones
    
    # dataframe of tickers, companies, cik
    df = pd.DataFrame({
        'Ticker': tickers,
        'Company': companies,
        # 'CIK': cik
    })

    return df

nasdaq_100_url = 'https://en.wikipedia.org/wiki/NASDAQ-100'
sp500_url = 'http://en.wikipedia.org/wiki/List_of_S%26P_500_companies'

In [12]:
# nasdaq_100_df = fetch_tickers(nasdaq_100_url, table=4, index_col="Ticker", company_col="Company")
sp500_df = fetch_tickers(sp500_url)
sp500_df

,Ticker,Company
0,MMM,3M
1,AOS,A. O. Smith
2,ABT,Abbott Laboratories
3,ABBV,AbbVie
4,ACN,Accenture
...,...,...
498,XYL,Xylem Inc.
499,YUM,Yum! Brands
500,ZBRA,Zebra Technologies
501,ZBH,Zimmer Biomet


In [13]:
gig_tickers = {
    'Upwork': "UPWK",
    'Fiverr': "FVRR",
    'Uber': "UBER",
    'Lyft': "LYFT",
    'Doordash': "DASH",
    'Grubhub': "GRUB",
    'Instacart': "CART",
    'Etsy': "ETSY",
    'Shopify': "SHOP",
    'Udemy': "UDMY",
    'Herbalife': "HLF",
    'Primerica': "PRI",
    'Tupperware': "TUP",
    'Avon': "AVP",
}

# etf_tickers = {
#     'S&P 500': 'SPY',
#     'Nasdaq 100': 'QQQ',
#     'Russell 2000': 'IWM'
# }

# nasdaq100_tickers = nasdaq_100_df.set_index('Company').to_dict()['Ticker']

sp500_tickers = sp500_df.set_index('Company').to_dict()['Ticker']


In [18]:
def load_yfinance_data(tickers):
    financial_data = []
    for company, ticker in tickers.items():
        try:
            stock = yf.Ticker(ticker)
            # period is 2005 thru 2025
            hist = stock.history(start="2005-01-01", end="2025-12-31")
            info = stock.info
            hist['Company'] = company
            hist['Ticker'] = ticker
            # print(info.keys())
            info = {key: info.get(key, None) for key in [
                # 'marketCap', 'totalRevenue', 'totalDebt', 'debtToEquity', 'profitMargins', 'grossProfits', 
                # 'grossMargins', 'ebitda', 'ebitdaMargins', 'operatingMargins', 'earningsGrowth', 'trailingEps', 
                # 'forwardEps', 'epsCurrentYear', 'fullTimeEmployees','auditRisk', 'boardRisk', 'compensationRisk', 
                # 'shareHolderRightsRisk', 'overallRisk', 'averageAnalystRating', 
                'industryKey', 'sectorKey', 
                # 'longBusinessSummary'
                ]}
            for key, value in info.items():
                hist[key] = value
            financial_data.append(hist)
        except Exception as e:
            print(f"Error fetching data for {company} ({ticker}): {e}")
    if financial_data:
        financial_data = pd.concat(financial_data)
        financial_data.reset_index(inplace=True)
        
        cols_to_drop = ['Dividends', 'Stock Splits', 'Adj Close', 'Open', 'High', 'Low']
         # Check if columns exist before dropping
        for col in cols_to_drop:
            if col in financial_data.columns:
                financial_data = financial_data.drop(columns=col)
            
        cols_to_reorder = ['sectorKey', 'industryKey', 'Ticker', 'Company', 'Date']
         # Check if columns exist before reordering
        for col in cols_to_reorder:
            if col in financial_data.columns:
                # Set as first columns
                financial_data.insert(0, col, financial_data.pop(col))
        
        # set Date as y-m-d format
        financial_data['Date'] = pd.to_datetime(financial_data['Date']).dt.date
        
        return financial_data
    else:
        return pd.DataFrame()

In [19]:
df = yf.download("GRUB", start="2016-01-01", end="2018-01-01")
df

[*********************100%***********************]  1 of 1 completed

1 Failed download:
- GRUB: No timezone found, symbol may be delisted


,Open,High,Low,Close,Adj Close,Volume
Date,,,,,,


In [ ]:
# gig_data = load_yfinance_data(gig_tickers) 

 # gig_data.to_csv('../../data/yfinance/gig_yfinance_raw.csv', index=False)
gig_data = pd.read_csv('../../data/yfinance/gig_yfinance_raw.csv')

gig_data

,Date,Company,Ticker,industryKey,sectorKey,Close,Volume
0,2018-10-03,Upwork,UPWK,internet-content-information,communication-services,21.180000,8385100.0
1,2018-10-04,Upwork,UPWK,internet-content-information,communication-services,20.959999,1112900.0
2,2018-10-05,Upwork,UPWK,internet-content-information,communication-services,20.990000,914000.0
3,2018-10-08,Upwork,UPWK,internet-content-information,communication-services,20.040001,797300.0
4,2018-10-09,Upwork,UPWK,internet-content-information,communication-services,19.940001,1244300.0
...,...,...,...,...,...,...,...
24328,2025-12-23,Primerica,PRI,insurance-life,financial-services,261.055237,135100.0
24329,2025-12-24,Primerica,PRI,insurance-life,financial-services,260.955688,109900.0
24330,2025-12-26,Primerica,PRI,insurance-life,financial-services,259.014618,174500.0
24331,2025-12-29,Primerica,PRI,insurance-life,financial-services,258.049072,187500.0


In [8]:
# nasdaq100_data = load_yfinance_data(nasdaq100_tickers)
# nasdaq100_data.to_csv('../../data/yfinance/raw/nasdaq100_data.csv', index=False)

# nasdaq100_data = pd.read_csv('../../data/yfinance/raw/nasdaq100_data.csv')
# nasdaq100_data

In [ ]:
# sp500_data = load_yfinance_data(sp500_tickers)

# sp500_data.to_csv('../../data/yfinance/sp500_yfinance_raw.csv', index=False)
sp500_data = pd.read_csv('../../data/yfinance/sp500_yfinance_raw.csv')

sp500_data

,Date,Company,Ticker,industryKey,sectorKey,Close,Volume
0,2005-01-03,3M,MMM,conglomerates,industrials,37.293541,3817632.0
1,2005-01-04,3M,MMM,conglomerates,industrials,36.990295,4358942.0
2,2005-01-05,3M,MMM,conglomerates,industrials,36.537708,3462779.0
3,2005-01-06,3M,MMM,conglomerates,industrials,36.868103,3605342.0
4,2005-01-07,3M,MMM,conglomerates,industrials,37.248268,3938428.0
...,...,...,...,...,...,...,...
2406766,2025-12-23,Zoetis,ZTS,drug-manufacturers-specialty-generic,healthcare,123.014725,4952000.0
2406767,2025-12-24,Zoetis,ZTS,drug-manufacturers-specialty-generic,healthcare,124.956429,2369000.0
2406768,2025-12-26,Zoetis,ZTS,drug-manufacturers-specialty-generic,healthcare,125.693283,3226800.0
2406769,2025-12-29,Zoetis,ZTS,drug-manufacturers-specialty-generic,healthcare,125.444351,4465600.0


In [10]:
# etf_data = load_yfinance_data(etf_tickers)
# etf_data.to_csv('../../data/yfinance/raw/etf_data.csv', index=False)

# etf_data = pd.read_csv('../../data/yfinance/raw/etf_data.csv')
# etf_data

In [ ]:
# Function to covert each finance dataset to different frequencies and save as csv
def save_resampled_data(data, prefix):
    # Ensure Date column is datetime
    df = data.copy()
    df['Date'] = pd.to_datetime(df['Date'])
    
    # drop columns with all nans
    df = df.dropna(axis=1, how='all')
    
    # create batch function for batches per Ticker to improve resampling performance
    tickers = df['Ticker'].unique()
    df_list = [df[df['Ticker'] == ticker] for ticker in tickers]
    
    weekly_list = []
    monthly_list = []
    quarterly_list = []
    annually_list = []
    
    # Define pct change periods for each frequency
    pct_periods = {
        '1w': 1,   # 1 week lag for weekly
        '1m': 4,  # 4 weeks in a month (approx)
        '3m': 13, # 13 weeks in 3 months (approx)
        '6m': 26, # 26 weeks in 6 months (approx)
        # '1y': 52  # 52 weeks in a year (approx)
    }
    
    for ticker_df in df_list:
        # Separate numeric and non-numeric columns
        numeric_cols = ticker_df.select_dtypes(include='number').columns.tolist()
        non_numeric_cols = ticker_df.select_dtypes(exclude=['number', 'datetime']).columns.tolist()
        
        # Build aggregation dict: last for all columns
        agg_dict = {col: 'last' for col in non_numeric_cols}
        agg_dict.update({col: 'last' for col in numeric_cols})
        
        # Resample - no need for groupby since we're already filtering by ticker
        weekly = ticker_df.resample('W', on='Date').agg(agg_dict).reset_index()
        monthly = ticker_df.resample('ME', on='Date').agg(agg_dict).reset_index()
        quarterly = ticker_df.resample('QE', on='Date').agg(agg_dict).reset_index()
        annually = ticker_df.resample('YE', on='Date').agg(agg_dict).reset_index()
        
        # Add all pct change columns to weekly (highest frequency) for Close and Volume
        for label, period in pct_periods.items():
            close_col = f'Close_pct_change_{label}'
            vol_col = f'Volume_pct_change_{label}'
            weekly[close_col] = weekly.groupby('Ticker')['Close'].pct_change(periods=period).fillna(0)
            weekly[vol_col] = weekly.groupby('Ticker')['Volume'].pct_change(periods=period).fillna(0)
        
        # Add only relevant pct change columns to lower frequencies for Close and Volume
        # Monthly
        monthly['Close_pct_change_1m'] = monthly.groupby('Ticker')['Close'].pct_change(periods=1).fillna(0)
        monthly['Close_pct_change_3m'] = monthly.groupby('Ticker')['Close'].pct_change(periods=3).fillna(0)
        monthly['Close_pct_change_6m'] = monthly.groupby('Ticker')['Close'].pct_change(periods=6).fillna(0)
        monthly['Volume_pct_change_1m'] = monthly.groupby('Ticker')['Volume'].pct_change(periods=1).fillna(0)
        monthly['Volume_pct_change_3m'] = monthly.groupby('Ticker')['Volume'].pct_change(periods=3).fillna(0)
        monthly['Volume_pct_change_6m'] = monthly.groupby('Ticker')['Volume'].pct_change(periods=6).fillna(0)
        # Quarterly
        quarterly['Close_pct_change_3m'] = quarterly.groupby('Ticker')['Close'].pct_change(periods=1).fillna(0)
        quarterly['Close_pct_change_6m'] = quarterly.groupby('Ticker')['Close'].pct_change(periods=2).fillna(0)
        quarterly['Volume_pct_change_3m'] = quarterly.groupby('Ticker')['Volume'].pct_change(periods=1).fillna(0)
        quarterly['Volume_pct_change_6m'] = quarterly.groupby('Ticker')['Volume'].pct_change(periods=2).fillna(0)
        # Annually
        annually['Close_pct_change_1y'] = annually.groupby('Ticker')['Close'].pct_change(periods=1).fillna(0)
        annually['Volume_pct_change_1y'] = annually.groupby('Ticker')['Volume'].pct_change(periods=1).fillna(0)
        
        weekly_list.append(weekly)
        monthly_list.append(monthly)
        quarterly_list.append(quarterly)
        annually_list.append(annually)
    
    # Concatenate all tickers
    data_weekly = pd.concat(weekly_list, ignore_index=True).dropna(subset=['Ticker'])
    data_monthly = pd.concat(monthly_list, ignore_index=True).dropna(subset=['Ticker'])
    data_quarterly = pd.concat(quarterly_list, ignore_index=True).dropna(subset=['Ticker'])
    data_annually = pd.concat(annually_list, ignore_index=True).dropna(subset=['Ticker'])
    
    all_data = [data_weekly, data_monthly, data_quarterly, data_annually]
    
    # Add week, month, quarter, year columns based on frequency
    for i, dataset in enumerate(all_data):
        if i == 0:  # weekly
            dataset['Week'] = dataset['Date'].dt.isocalendar().week
            dataset['Month'] = dataset['Date'].dt.month
            dataset['Quarter'] = dataset['Date'].dt.quarter
            dataset['Year'] = dataset['Date'].dt.year
        elif i == 1:  # monthly
            dataset['Month'] = dataset['Date'].dt.month
            dataset['Quarter'] = dataset['Date'].dt.quarter
            dataset['Year'] = dataset['Date'].dt.year
        elif i == 2:  # quarterly
            dataset['Quarter'] = dataset['Date'].dt.quarter
            dataset['Year'] = dataset['Date'].dt.year
        else:  # annually
            dataset['Year'] = dataset['Date'].dt.year
        
    # Move frequency columns to beginning
    for i, dataset in enumerate(all_data):
        cols = dataset.columns.tolist()
        time_cols = ['Date', 'Week', 'Month', 'Quarter', 'Year']
        existing_time_cols = [col for col in time_cols if col in cols]
        new_cols = existing_time_cols + [col for col in cols if col not in time_cols]
        all_data[i] = dataset[new_cols]
        
    # all_data[0].to_csv(f'../../data/yfinance/{prefix}_yfinance_weekly.csv', index=False)
    all_data[1].to_csv(f'../../data/yfinance/{prefix}_yfinance_monthly.csv', index=False)
    all_data[2].to_csv(f'../../data/yfinance/{prefix}_yfinance_quarterly.csv', index=False)
    all_data[3].to_csv(f'../../data/yfinance/{prefix}_yfinance_annually.csv', index=False)
    
save_resampled_data(gig_data, 'gig_data')
save_resampled_data(sp500_data, 'sp500_data')

In [52]:
# Example of loading the monthly data back in
gig_monthly = pd.read_csv('../../data/yfinance/gig_data_monthly.csv')
gig_monthly

,Date,Month,Quarter,Year,Company,Ticker,industryKey,sectorKey,Close,Volume,Close_pct_change_1m,Close_pct_change_3m,Close_pct_change_6m,Volume_pct_change_1m,Volume_pct_change_3m,Volume_pct_change_6m
0,2018-10-31,10,4,2018,Upwork,UPWK,internet-content-information,communication-services,19.200001,337600.0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,2018-11-30,11,4,2018,Upwork,UPWK,internet-content-information,communication-services,18.629999,352700.0,-0.029688,0.000000,0.000000,0.044727,0.000000,0.000000
2,2018-12-31,12,4,2018,Upwork,UPWK,internet-content-information,communication-services,18.110001,337200.0,-0.027912,0.000000,0.000000,-0.043947,0.000000,0.000000
3,2019-01-31,1,1,2019,Upwork,UPWK,internet-content-information,communication-services,19.309999,368200.0,0.066262,0.005729,0.000000,0.091934,0.090640,0.000000
4,2019-02-28,2,1,2019,Upwork,UPWK,internet-content-information,communication-services,23.690001,447200.0,0.226826,0.271605,0.000000,0.214557,0.267933,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1161,2025-08-31,8,3,2025,Primerica,PRI,insurance-life,financial-services,267.036713,157200.0,0.017960,-0.000736,-0.064006,-0.233918,-0.015038,-0.006321
1162,2025-09-30,9,3,2025,Primerica,PRI,insurance-life,financial-services,275.216156,145800.0,0.030630,0.018318,-0.016791,-0.072519,-0.636862,-0.552211
1163,2025-10-31,10,4,2025,Primerica,PRI,insurance-life,financial-services,257.647675,230200.0,-0.063835,-0.017831,-0.000670,0.578875,0.121832,-0.122713
1164,2025-11-30,11,4,2025,Primerica,PRI,insurance-life,financial-services,256.137909,55900.0,-0.005860,-0.040814,-0.041520,-0.757168,-0.644402,-0.649749
